# Data cleaning and pre-processing

## Missing values

In [ ]:
"""
Comparing missing-value treatment methods.

Setup: a synthetic dataset with a known linear relationship
    y = 2*x1 + 3*x2 + noise
Missing values are introduced into x2 only (MCAR by default: missing
completely at random, independent of everything; optionally MAR: missing
more often when x1 is small, i.e. dependent on an OBSERVED variable).
A separate, fully observed test set (no missingness at all) is used to
evaluate downstream predictive performance honestly.

Each method is compared on three things:
    1. Bias in the recovered mean/variance of x2 itself, and the true
       (pre-missingness) mean/variance rendering mean imputation's
       variance-shrinking effect visible.
    2. Bias in the fitted regression coefficient on x2 (true value = 3),
       fit via ordinary least squares (closed form) on each treated dataset.
    3. Predictive MSE on the clean, fully observed held-out test set.

Methods compared:
    - Listwise deletion             (statistics tradition; valid under MCAR)
    - Mean imputation               (both traditions; discouraged in
                                      statistics as a primary method)
    - Missingness indicator + mean   (ML tradition)
    - k-NN imputation                (ML tradition)
    - Iterative / MICE-style multiple imputation (statistics tradition;
                                      M=20 posterior draws pooled by
                                      averaging, following Rubin's rules
                                      for the point estimate)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

N_TRAIN = 500
N_TEST = 2000
TRUE_W = np.array([2.0, 3.0])   # true coefficients on [x1, x2]
NOISE_STD = 1.0
MISSING_RATE = 0.3
MECHANISM = "MCAR"   # "MCAR" or "MAR"
SEED = 0


def generate_full_data(n, rng):
    x1 = rng.normal(0, 1, n)
    x2 = 0.5 * x1 + rng.normal(0, 1, n)   # x1, x2 correlated, as in real data
    y = TRUE_W[0] * x1 + TRUE_W[1] * x2 + rng.normal(0, NOISE_STD, n)
    return pd.DataFrame({"x1": x1, "x2": x2, "y": y})


def introduce_missingness(df, rate, mechanism, rng):
    df = df.copy()
    n = len(df)
    if mechanism == "MCAR":
        missing_mask = rng.uniform(size=n) < rate
    elif mechanism == "MAR":
        # missing more often when x1 is small - depends only on the
        # OBSERVED variable x1, not on x2's own (unobserved) value.
        prob = 1 / (1 + np.exp(3 * (df["x1"] - df["x1"].median())))
        prob = prob / prob.mean() * rate
        prob = np.clip(prob, 0, 1)
        missing_mask = rng.uniform(size=n) < prob
    else:
        raise ValueError(mechanism)
    df.loc[missing_mask, "x2"] = np.nan
    return df


def fit_ols(X, y):
    """Closed-form OLS via the normal equations applied here to a design matrix with a bias column."""
    X_design = np.column_stack([np.ones(len(X)), X])
    w = np.linalg.solve(X_design.T @ X_design, X_design.T @ y)
    return w[1:], w[0]  # coefficients, intercept


def evaluate(name, df_train, X_test, y_test, true_x2_mean, true_x2_var):
    w, b = fit_ols(df_train[["x1", "x2"]].to_numpy(), df_train["y"].to_numpy())
    y_pred = X_test @ w + b
    test_mse = np.mean((y_pred - y_test) ** 2)

    x2_mean = df_train["x2"].mean()
    x2_var = df_train["x2"].var()

    return {
        "method": name,
        "n_used": len(df_train),
        "x2_mean": x2_mean,
        "x2_mean_bias": x2_mean - true_x2_mean,
        "x2_var": x2_var,
        "x2_var_ratio": x2_var / true_x2_var,
        "coef_x1": w[0],
        "coef_x2": w[1],
        "coef_x2_bias": w[1] - TRUE_W[1],
        "test_mse": test_mse,
    }


def main():
    rng = np.random.default_rng(SEED)
    rng_test = np.random.default_rng(SEED + 1)

    df_full = generate_full_data(N_TRAIN, rng)          # ground truth, pre-missingness
    df_test = generate_full_data(N_TEST, rng_test)       # clean, held-out
    X_test = df_test[["x1", "x2"]].to_numpy()
    y_test = df_test["y"].to_numpy()

    true_x2_mean = df_full["x2"].mean()
    true_x2_var = df_full["x2"].var()

    df_missing = introduce_missingness(df_full, MISSING_RATE, MECHANISM, rng)
    n_missing = df_missing["x2"].isna().sum()
    print(f"Mechanism: {MECHANISM}, target rate: {MISSING_RATE}, "
          f"actual missing: {n_missing}/{N_TRAIN} ({n_missing/N_TRAIN:.1%})\n")

    results = []

    # Oracle: the true, pre-missingness data (for reference only - not a
    # real-world option, since in practice you never observe this).
    results.append(evaluate("Oracle (no missingness)", df_full, X_test, y_test,
                             true_x2_mean, true_x2_var))

    # 1. Listwise deletion.
    df_listwise = df_missing.dropna(subset=["x2"])
    results.append(evaluate("Listwise deletion", df_listwise, X_test, y_test,
                             true_x2_mean, true_x2_var))

    # 2. Mean imputation.
    df_mean = df_missing.copy()
    mean_imputer = SimpleImputer(strategy="mean")
    df_mean["x2"] = mean_imputer.fit_transform(df_mean[["x2"]])
    results.append(evaluate("Mean imputation", df_mean, X_test, y_test,
                             true_x2_mean, true_x2_var))

    # 3. Missingness indicator + mean imputation.
    df_indicator = df_mean.copy()
    df_indicator["x2_missing"] = df_missing["x2"].isna().astype(float)
    w_ind, b_ind = fit_ols(df_indicator[["x1", "x2", "x2_missing"]].to_numpy(),
                            df_indicator["y"].to_numpy())
    y_pred_ind = np.column_stack([X_test, np.zeros(len(X_test))]) @ w_ind + b_ind
    results.append({
        "method": "Missingness indicator + mean",
        "n_used": len(df_indicator),
        "x2_mean": df_indicator["x2"].mean(),
        "x2_mean_bias": df_indicator["x2"].mean() - true_x2_mean,
        "x2_var": df_indicator["x2"].var(),
        "x2_var_ratio": df_indicator["x2"].var() / true_x2_var,
        "coef_x1": w_ind[0], "coef_x2": w_ind[1], "coef_x2_bias": w_ind[1] - TRUE_W[1],
        "test_mse": np.mean((y_pred_ind - y_test) ** 2),
    })

    # 4. k-NN imputation (uses x1 to inform the imputed x2 value).
    df_knn = df_missing.copy()
    knn_imputer = KNNImputer(n_neighbors=10)
    df_knn[["x1", "x2"]] = knn_imputer.fit_transform(df_knn[["x1", "x2"]])
    results.append(evaluate("k-NN imputation", df_knn, X_test, y_test,
                             true_x2_mean, true_x2_var))

    # 5. Multiple Imputation (proper MI): draw M completed datasets from
    # the IterativeImputer's posterior, fit OLS on each, and POOL the
    # resulting coefficients and predictions by simple averaging (Rubin's
    # rules for the point estimate). A single posterior draw is not yet
    # "multiple imputation" - pooling across draws is the actual method.
    M = 20
    mi_coefs, mi_preds, mi_x2_means, mi_x2_vars = [], [], [], []
    for m in range(M):
        imputer = IterativeImputer(random_state=SEED + m, sample_posterior=True)
        df_m = df_missing.copy()
        # Include y in the imputation model itself (standard MI practice:
        # when the completed data will be used to fit a model involving
        # y, omitting y from the imputation model attenuates the imputed
        # predictor's relationship with y, biasing the downstream
        # coefficient toward zero).
        imputed = imputer.fit_transform(df_m[["x1", "x2", "y"]])
        df_m["x2"] = imputed[:, 1]
        w_m, b_m = fit_ols(df_m[["x1", "x2"]].to_numpy(), df_m["y"].to_numpy())
        mi_coefs.append(w_m)
        mi_preds.append(X_test @ w_m + b_m)
        mi_x2_means.append(df_m["x2"].mean())
        mi_x2_vars.append(df_m["x2"].var())

    mi_coef_pooled = np.mean(mi_coefs, axis=0)
    mi_pred_pooled = np.mean(mi_preds, axis=0)   # pooled prediction, per Rubin's rules
    mi_x2_mean_pooled = np.mean(mi_x2_means)
    mi_x2_var_pooled = np.mean(mi_x2_vars)

    results.append({
        "method": f"Multiple Imputation (M={M}, pooled)",
        "n_used": len(df_missing),
        "x2_mean": mi_x2_mean_pooled,
        "x2_mean_bias": mi_x2_mean_pooled - true_x2_mean,
        "x2_var": mi_x2_var_pooled,
        "x2_var_ratio": mi_x2_var_pooled / true_x2_var,
        "coef_x1": mi_coef_pooled[0], "coef_x2": mi_coef_pooled[1],
        "coef_x2_bias": mi_coef_pooled[1] - TRUE_W[1],
        "test_mse": np.mean((mi_pred_pooled - y_test) ** 2),
    })

    results_df = pd.DataFrame(results).set_index("method")
    pd.set_option("display.float_format", lambda v: f"{v:.4f}")
    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 20)
    print(results_df[["n_used", "x2_mean_bias", "x2_var_ratio",
                       "coef_x2", "coef_x2_bias", "test_mse"]])

    # --- Plot: coefficient bias and test MSE per method ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
    methods = results_df.index.tolist()
    colors = ["grey"] + ["tab:blue"] * (len(methods) - 1)

    ax1.barh(methods, results_df["coef_x2"], color=colors)
    ax1.axvline(TRUE_W[1], color="red", linestyle="--", label=f"true coefficient = {TRUE_W[1]}")
    ax1.set_xlabel("fitted coefficient on x2")
    ax1.set_title("Bias in the regression coefficient")
    ax1.legend(fontsize=8)
    ax1.invert_yaxis()

    ax2.barh(methods, results_df["test_mse"], color=colors)
    ax2.set_xlabel("test MSE (clean held-out set)")
    ax2.set_title("Predictive performance")
    ax2.invert_yaxis()

    fig.suptitle(f"Missing-value treatment comparison "
                 f"(mechanism={MECHANISM}, missing rate={MISSING_RATE})")
    fig.tight_layout()
    fig.savefig("missing_value_comparison.png", dpi=150)
    print("\nSaved plot to missing_value_comparison.png")


if __name__ == "__main__":
    main()

## Text pre-processing

In [ ]:
"""
Text analysis demo, following the pre-processing pipeline introduced in
the lecture notes on Natural Language Processing (Section 12):

    Step 1: Scrape titles and abstracts of papers listed under
            https://research.google/pubs/?category=algorithms-and-theory
            (1400+ papers across ~96 pages at the time of writing)
    Step 2: Tokenisation
    Step 3: Cleaning (lowercasing, punctuation/URL/@-mention removal)
    Step 4: Stopword removal
    Step 5: Stemming vs. lemmatisation (compared side by side)
    Step 6: Bag-of-words representation
    Step 7: N-grams
    Step 8: Vocabulary construction, pruning
    Step 9: TF-IDF weighting
    Step 10: Cosine similarity between documents (the application from
             the lecture's Bag-of-words subsection)

Requires: requests, beautifulsoup4, nltk, scikit-learn, matplotlib.
Optional, for full multi-page scraping: playwright
    (pip install playwright && playwright install chromium)

NOTE ON PAGINATION: this listing shows only 15 papers per page, and its
"page 2", "page 3", ... links are JavaScript-driven (they do not point
to a separate URL that can be requested directly) - the page content is
swapped client-side. A plain requests+BeautifulSoup GET can therefore
only ever retrieve the FIRST page (~15 papers). To get "lots of papers"
across many pages, this script uses Playwright (a scriptable
browser) to click through the pagination controls, which are reliably
identifiable by their title attribute in the rendered page ("Go to page
2", "Go to page 3", ...), and re-scrapes the DOM after each click.

If Playwright (or its browser binary) is not installed, the script
falls back to a single-page requests-based scrape (~15 papers), and if
even that fails (no internet access at all, or the page structure has
changed), it falls back further to a small embedded set of real
title/abstract pairs captured from the page at the time of writing, so
the rest of the pipeline can still be demonstrated end to end.
"""

import re
import time
import warnings

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction import text as sk_text
from sklearn.metrics.pairwise import cosine_similarity

BASE_URL = "https://research.google/pubs/?category=algorithms-and-theory"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; AppliedML-course-demo/1.0)"}
MAX_PAGES = 10          # ~15 papers/page -> up to ~150 papers
PAGE_LOAD_WAIT_S = 2.0   # settle time after each pagination click

In [ ]:
# Fallback data: real title/abstract pairs captured from page 1 of the
# listing, used only if BOTH the Playwright and the plain-requests
# scraping paths fail (see the note above).
FALLBACK_PAPERS = [
    {"title": "Marginalized Bundle Adjustment: Multi-View Camera Pose from "
              "Monocular Depth Estimates",
     "abstract": "This paper looks at recovering camera positions and 3D "
                  "scene layout from multiple photos, a long-studied "
                  "computer vision problem. The authors combine modern "
                  "single-image depth prediction with the classical "
                  "bundle-adjustment approach, introducing a way to account "
                  "for how noisy these depth estimates can be. Their method "
                  "performs competitively across setups ranging from just "
                  "two photos to thousands."},
    {"title": "Geo-Contextual AI Concierge: Minimizing Internal Support "
              "Friction Through Proactive Knowledge Synthesis",
     "abstract": "Large organisations often struggle because internal "
                  "documentation goes stale faster than local teams can "
                  "update it, pushing employees toward costly help-desk "
                  "tickets instead of self-service. The authors describe a "
                  "system with two cooperating AI components: one that "
                  "scans past support cases to flag where documentation is "
                  "weak, and another that answers employee questions "
                  "on the spot using location-aware retrieval."},
    {"title": "Mind the Gap: Structure-Aware Consistency in Preference "
              "Learning",
     "abstract": "Methods for aligning language models with human "
                  "preferences, including DPO, typically optimise a proxy "
                  "loss rather than the true ranking objective. The authors "
                  "show this proxy can fail to guarantee good ranking "
                  "behaviour for realistic neural network models, and "
                  "prove that requiring a confidence margin between "
                  "preferred and rejected responses is essential, proposing "
                  "a margin that adapts to how semantically different two "
                  "responses are."},
    {"title": "A Theoretical Framework for Modular Learning of Robust "
              "Generative Models",
     "abstract": "Training one enormous language model on a mixed dataset "
                  "usually depends on ad hoc weighting choices. This paper "
                  "asks whether several smaller, specialised models can "
                  "instead be combined to match a single large model's "
                  "performance, without needing to hand-tune that mixture, "
                  "and proves that a suitably designed combination "
                  "mechanism can guarantee robust performance regardless "
                  "of the data mix."},
    {"title": "Beyond Tsybakov: Model Margin Noise and H-Consistency "
              "Bounds",
     "abstract": "Classification theory often relies on the Tsybakov noise "
                  "condition to bound how hard a classification problem "
                  "is. The authors propose a related but weaker condition, "
                  "tied to how far a specific classifier is from the ideal "
                  "one rather than a fixed property of the data, and show "
                  "it still yields strong theoretical guarantees for both "
                  "binary and multi-class problems."},
    {"title": "A simple and efficient implementation of strong call by "
              "need by an abstract machine",
     "abstract": "This is a programming-languages paper about efficiently "
                  "evaluating lazy functional programs. The authors "
                  "mechanically derive a new abstract machine for a strong "
                  "call-by-need evaluation strategy from an existing "
                  "higher-order interpreter, and prove it behaves "
                  "correctly and efficiently compared to earlier known "
                  "machines for related, weaker strategies."},
    {"title": "Meridian GeoX: An Open-Source Framework for Precision and "
              "Efficiency in Geo Experiments",
     "abstract": "Geo experiments, which compare outcomes across "
                  "geographic regions, are a privacy-friendly way to "
                  "measure whether advertising works, but they are "
                  "expensive to run reliably because regions vary so much "
                  "and standard statistical methods struggle with "
                  "real-world trends over time. The authors introduce an "
                  "open-source toolkit offering several improved "
                  "statistical designs that reduce the required budget "
                  "while keeping false-positive rates under control."},
    {"title": "Surviving the perfect storm: how hardware PMs can beat the "
              "AI tax and trade tariffs",
     "abstract": "This is a practitioner-oriented piece for hardware "
                  "product managers, addressing how rising AI-related "
                  "compute costs and trade tariffs squeeze consumer "
                  "device margins. It lays out a joint-development "
                  "partnership approach as a way to keep innovating "
                  "without letting these external cost pressures stall "
                  "product roadmaps."},
    {"title": "One Attack to Rule Them All: Tight Quadratic Bounds for "
              "Adaptive Queries on Cardinality Sketches",
     "abstract": "Cardinality sketches are small data structures that "
                  "estimate how many distinct items are in a dataset and "
                  "can be merged across sets. This paper shows that a "
                  "broad family of such sketches can be broken by an "
                  "attacker who adapts each query based on earlier "
                  "answers, using far fewer queries than previously "
                  "thought necessary, and pins down the exact number of "
                  "queries required for several important special cases."},
    {"title": "Rational Area Profiles of Regular Polygons Cut by Two "
              "Congruent Diagonals",
     "abstract": "This is a paper in geometric number theory. Drawing two "
                  "equal-length crossing diagonals inside a regular "
                  "polygon splits it into four regions, and the authors "
                  "work out exactly when the areas of these regions (or "
                  "sums of them) come out as rational numbers, for every "
                  "possible polygon and choice of diagonals, using tools "
                  "from the algebra of roots of unity."},
    {"title": "Load Balancing under Adaptive Bin Deletions",
     "abstract": "The authors study a variant of the classic balls-and-bins "
                  "problem where an adversary repeatedly removes bins, "
                  "forcing whatever balls were inside to be moved "
                  "elsewhere. They show that simply redistributing those "
                  "balls at random keeps both the movement cost and the "
                  "worst-case bin load close to optimal, and that a "
                  "well-known trick of checking two candidate bins instead "
                  "of one improves the load balance further."},
    {"title": "A Simple and Robust Protocol for Distributed Counting",
     "abstract": "Distributed counting asks many sites to jointly track a "
                  "running total while sending as few messages as "
                  "possible. Earlier work showed randomisation helps a lot, "
                  "but only under the assumption that the data isn't "
                  "adversarially chosen in response to the protocol's own "
                  "outputs; this paper shows the standard randomised "
                  "protocol actually breaks under such an adversary, and "
                  "gives a new protocol that is both simpler and provably "
                  "resistant to it."},
    {"title": "Adaptively Robust Resettable Streaming",
     "abstract": "The authors consider streaming data where a running "
                  "count for a key can be increased or wiped back to zero, "
                  "a pattern relevant to monitoring systems that must "
                  "support deletion and to machine-unlearning use cases. "
                  "They show existing compact summaries for this setting "
                  "can be manipulated by an adaptive attacker, and design "
                  "new summaries, protected using differential privacy "
                  "techniques, that resist this while still using very "
                  "little memory."},
    {"title": "Rate-Distortion Optimized LoRA for Efficient Post-Filtering "
              "in AV2",
     "abstract": "This paper is about improving video compression quality "
                  "using a lightweight, adaptable neural filter for the "
                  "upcoming AV2 video standard. Rather than sending a full "
                  "new neural network for each video, the authors update "
                  "only a small set of low-rank parameters per video and "
                  "carefully budget how many of these parameters each part "
                  "of the network gets, improving compression efficiency "
                  "at a low added cost."},
    {"title": "Overview of the Block-Partitioning Framework in AV2",
     "abstract": "This paper describes how the AV2 video codec decides "
                  "the size of the image blocks it uses for prediction and "
                  "compression, a design choice with a major effect on "
                  "compression efficiency. It walks through the redesigned "
                  "partitioning rules, including a new option that lets "
                  "colour and brightness information be split into blocks "
                  "independently, and reports on which design choices "
                  "matter most."},
]

In [ ]:
# ---------------------------------------------------------------------
# Step 1: Scraping (multi-page, via Playwright; falls back gracefully)
# ---------------------------------------------------------------------

def _extract_papers_from_html(html):
    """Given one page's rendered HTML, extract (title, abstract) pairs
    using the same text-pattern approach as before: titles are the
    /pubs/... links whose visible text isn't the generic "View details"
    label, and abstracts follow "Preview abstract ... View details" in
    the page's rendered text, in document order."""
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html, "html.parser")
    pub_links = soup.find_all("a", href=re.compile(r"/pubs/"))
    titles, seen = [], set()
    for link in pub_links:
        label = link.get_text(strip=True)
        if not label or label.lower() == "view details":
            continue
        if label not in seen:
            seen.add(label)
            titles.append(label)

    full_text = soup.get_text(" ", strip=True)
    abstracts = re.findall(
        r"Preview abstract\s+(.*?)\s+View details", full_text, flags=re.DOTALL
    )

    n = min(len(titles), len(abstracts))
    return [{"title": titles[i], "abstract": abstracts[i]} for i in range(n)]


def scrape_papers_playwright(url=BASE_URL, max_pages=MAX_PAGES):
    """Scrape multiple pages of the (JavaScript-paginated) publications
    listing using a real headless browser. Clicks through pagination
    controls identified by their title attribute ("Go to page N"),
    re-scraping the DOM after each click."""
    from playwright.sync_api import sync_playwright

    all_papers, seen_titles = [], set()
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page(user_agent=HEADERS["User-Agent"])
        page.goto(url, timeout=30000)
        page.wait_for_timeout(int(PAGE_LOAD_WAIT_S * 1000))

        for page_num in range(1, max_pages + 1):
            papers = _extract_papers_from_html(page.content())
            new_count = 0
            for paper in papers:
                if paper["title"] not in seen_titles:
                    seen_titles.add(paper["title"])
                    all_papers.append(paper)
                    new_count += 1
            print(f"  page {page_num}: {new_count} new papers "
                  f"(running total: {len(all_papers)})")

            if page_num == max_pages:
                break

            next_page_locator = page.get_by_title(f"Go to page {page_num + 1}")
            if next_page_locator.count() == 0:
                print(f"  no link to page {page_num + 1} found; stopping.")
                break
            next_page_locator.first.click()
            page.wait_for_timeout(int(PAGE_LOAD_WAIT_S * 1000))

        browser.close()
    return all_papers


def scrape_papers_single_page(url=BASE_URL):
    """Fallback: a plain GET retrieves only the first rendered page of
    the listing (~15 papers), since pagination beyond that is
    JavaScript-driven and not reachable via a URL parameter."""
    import requests

    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    papers = _extract_papers_from_html(resp.text)
    if not papers:
        raise ValueError("No title/abstract pairs found on the page.")
    return papers


def scrape_papers(url=BASE_URL, max_pages=MAX_PAGES):
    try:
        print("Attempting multi-page scrape via Playwright...")
        papers = scrape_papers_playwright(url, max_pages)
        if papers:
            print(f"Scraped {len(papers)} papers across up to {max_pages} pages.")
            return papers
        raise ValueError("Playwright scrape returned no papers.")
    except Exception as exc:
        warnings.warn(
            f"Playwright scraping unavailable or failed ({exc}); falling "
            f"back to a single-page requests-based scrape (~15 papers). "
            f"For full multi-page scraping, run: pip install playwright "
            f"&& playwright install chromium"
        )
    try:
        papers = scrape_papers_single_page(url)
        print(f"Scraped {len(papers)} papers (first page only).")
        return papers
    except Exception as exc:
        warnings.warn(f"Live scraping failed entirely ({exc}); using embedded "
                       f"fallback data ({len(FALLBACK_PAPERS)} papers).")
        return list(FALLBACK_PAPERS)

papers = scrape_papers()
documents = [f"{p['title']}. {p['abstract']}" for p in papers]
print(f"\n{len(documents)} documents to process.\n")


In [ ]:
# ---------------------------------------------------------------------
# Step 2-3: Tokenisation and cleaning
# ---------------------------------------------------------------------

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
TOKEN_RE = re.compile(r"[a-zA-Z]+")  # keeps letter-only tokens


def clean_and_tokenize(text):
    """Lowercase, strip URLs/@-mentions, then tokenize into words,
    discarding punctuation and digits (kept simple for this demo -
    a production pipeline might keep numbers or hyphenated terms)."""
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    return TOKEN_RE.findall(text)

# ---------------------------------------------------------------------
# Step 4: Stopword removal
# ---------------------------------------------------------------------

STOPWORDS = sk_text.ENGLISH_STOP_WORDS  # ships with scikit-learn, no download needed


def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS]

processed_tokens = []
for doc in documents:
    tokens = clean_and_tokenize(doc)
    tokens = remove_stopwords(tokens)
    processed_tokens.append(tokens)

print("Example (paper 1) after tokenisation + cleaning + stopword removal:")
print(processed_tokens[0][:20], "...\n")


In [ ]:
# ---------------------------------------------------------------------
# Step 5: Stemming vs. lemmatisation
# ---------------------------------------------------------------------

def get_stemmer_and_lemmatizer():
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()

    lemmatizer = None
    try:
        from nltk.stem import WordNetLemmatizer
        import nltk
        nltk.data.find("corpora/wordnet")
        lemmatizer = WordNetLemmatizer()
    except Exception:
        try:
            import nltk
            nltk.download("wordnet", quiet=True)
            nltk.download("omw-1.4", quiet=True)
            from nltk.stem import WordNetLemmatizer
            lemmatizer = WordNetLemmatizer()
        except Exception:
            warnings.warn(
                "WordNet corpus unavailable (no internet access to download "
                "it) - lemmatisation will be skipped; stemming still runs, "
                "since PorterStemmer needs no downloaded data."
            )
    return stemmer, lemmatizer

stemmer, lemmatizer = get_stemmer_and_lemmatizer()
sample_words = ["algorithms", "sketches", "queries", "classification",
                     "estimators", "better", "running"]
print("Stemming vs. lemmatisation, on a few sample words:")
print(f"{'word':<15}{'stem':<15}{'lemma'}")
for w in sample_words:
    stem = stemmer.stem(w)
    lemma = lemmatizer.lemmatize(w) if lemmatizer else "(unavailable)"
    print(f"{w:<15}{stem:<15}{lemma}")
print()

stemmed_docs = [" ".join(stemmer.stem(t) for t in tokens) for tokens in processed_tokens]


In [ ]:
# --- Step 6: bag-of-words ------------------------------------------------
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(stemmed_docs)
vocab = count_vectorizer.get_feature_names_out()
print(f"Bag-of-words: {len(vocab)} distinct terms across {len(documents)} documents.")

term_totals = np.asarray(bow_matrix.sum(axis=0)).ravel()
top_terms_idx = term_totals.argsort()[::-1][:10]
print("Top 10 most frequent terms (after stemming/stopword removal):")
for i in top_terms_idx:
    print(f"  {vocab[i]:<15} {term_totals[i]}")
print()


In [ ]:
# --- Step 7: N-grams -------------------------------------------------
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2), min_df=1)
bigram_matrix = bigram_vectorizer.fit_transform(stemmed_docs)
bigram_vocab = bigram_vectorizer.get_feature_names_out()
bigram_totals = np.asarray(bigram_matrix.sum(axis=0)).ravel()
top_bigrams_idx = bigram_totals.argsort()[::-1][:10]
print("Top 10 most frequent bigrams:")
for i in top_bigrams_idx:
    print(f"  {bigram_vocab[i]:<25} {bigram_totals[i]}")
print()


In [ ]:
# --- Step 8: vocabulary pruning -----------------------------------------
pruned_vectorizer = CountVectorizer(min_df=2, max_df=0.8)
pruned_matrix = pruned_vectorizer.fit_transform(stemmed_docs)
print(f"Vocabulary size before pruning: {len(vocab)}")
print(f"Vocabulary size after pruning (min_df=2, max_df=0.8): "
      f"{len(pruned_vectorizer.get_feature_names_out())}\n")


In [ ]:
# --- Step 9: TF-IDF -----------------------------------------------------
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(stemmed_docs)
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

print("Top 5 TF-IDF terms for each document:")
for i, p in enumerate(papers):
    row = tfidf_matrix[i].toarray().ravel()
    top_idx = row.argsort()[::-1][:5]
    top_terms = [(tfidf_vocab[j], round(row[j], 3)) for j in top_idx if row[j] > 0]
    print(f"  \"{p['title'][:60]}...\"")
    print(f"{top_terms}")
print()


In [ ]:
# --- Step 10: cosine similarity ------------------------------------------
sim_matrix = cosine_similarity(tfidf_matrix)
print("Cosine similarity matrix (TF-IDF vectors):")
print(np.round(sim_matrix, 2))

n = len(papers)
most_similar_pair, best_score = None, -1
for i in range(n):
    for j in range(i + 1, n):
        if sim_matrix[i, j] > best_score:
            best_score = sim_matrix[i, j]
            most_similar_pair = (i, j)
i, j = most_similar_pair
print(f"\nMost similar pair: \"{papers[i]['title'][:50]}...\" <-> "
      f"\"{papers[j]['title'][:50]}...\" (cosine similarity = {best_score:.3f})")

# --- Plot: similarity heatmap + top term frequencies ----------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im = ax1.imshow(sim_matrix, cmap="Greys", vmin=0, vmax=1)
ax1.set_xticks(range(n))
ax1.set_yticks(range(n))
ax1.set_xticklabels([f"P{k+1}" for k in range(n)])
ax1.set_yticklabels([f"P{k+1}" for k in range(n)])
ax1.set_title("Cosine similarity between papers (TF-IDF)")
fig.colorbar(im, ax=ax1, fraction=0.046)

ax2.barh([vocab[i] for i in top_terms_idx[::-1]], term_totals[top_terms_idx[::-1]],
      color="steelblue")
ax2.set_xlabel("total count across corpus")
ax2.set_title("Top 10 most frequent terms")

fig.tight_layout()
fig.savefig("text_analysis_demo.png", dpi=150)
print("\nSaved plot to text_analysis_demo.png")